[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Host_Prog/Intro_Databases/Intro_Databases.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Intro to Databases

Every experiment you run produces data that outlives the script that made it. This workshop is about *keeping* that data queryable: relational modeling, SQL, and talking to a database from Python — using a sensor-logging scenario you'd actually meet in a signals lab. Everything runs on `sqlite3` from Python's standard library: zero installation.

## 0. Introduction

Why not CSV files? They rot: no types, no relationships, no protection against half-written rows, and every question becomes a bespoke parsing script. A relational database gives you **structure** (tables with types), **integrity** (constraints, transactions), and a **query language** (SQL) that answers questions you hadn't thought of when you designed the file.

## 1. Pre-requisites

- [Intro to Python](../../Intro_Programming/Intro_Python/Intro_Python.ipynb) — functions, dicts, `with` blocks.
- Nothing else: `sqlite3` ships with Python.

---
### 🕐 Session 1 of 3 — *The Relational Model & SQL Basics* (~35 min)
**Goal:** design tables with keys and constraints; insert and query with SELECT/WHERE/ORDER BY.
**Feeds into:** Session 2 (joins & aggregation).

---

<details><summary>🎓 <b>Teacher notes — Session 1: The Relational Model & SQL Basics</b></summary>

**Motivate before syntax: why not CSV?** Students have all used CSVs and hit the failure modes — a column silently becomes text because one row has a typo, a script half-writes a file and corrupts it, "what's the average RMS for the microphone sensor" requires writing a bespoke parser every time. A relational database is a targeted fix for exactly those three pains: types, atomicity, and a general-purpose query language. Land this before touching SQL syntax, or the workshop reads as "here is a new dialect" instead of "here is why this dialect exists."

**The schema cell is doing more work than its size suggests — read it line by line.** `PRIMARY KEY` gives identity; `NOT NULL` and `CHECK` enforce invariants at write time, not read time; the `sensor_id INTEGER NOT NULL REFERENCES sensors(sensor_id)` line is a foreign key — call it out explicitly as the C-pointer analogy already in the intuition cell, except the database *refuses to let it dangle* (Session 1's constraint-rejection cell proves this). Also flag `PRAGMA foreign_keys = ON` as a genuine SQLite gotcha: foreign-key enforcement is opt-in per connection, unlike PostgreSQL/MySQL where it's always on — a student who forgets this pragma in their own project will see constraint violations silently succeed.

**The rejected-inserts cell is the payoff for the schema — don't skip it.** Three different guarantees fail three different ways: a `CHECK` catches a bad category value, `UNIQUE` catches a duplicate name, and the foreign key catches a `sensor_id` that doesn't exist yet. Ask students to predict *which* constraint each bad insert will trip before running the cell — that's a good check for whether the schema actually landed.

**SELECT/WHERE/ORDER BY is intentionally light here** — it's the one part of SQL every student has likely half-seen before. Don't over-invest time; the constraint and schema-design material is the differentiator.
</details>

## 2. Tables, Keys, Constraints

💡 **Intuition.** A table is a *typed spreadsheet with rules*. The **primary key** gives every row an identity; a **foreign key** is a pointer from one table's rows to another's — the relational version of the C pointer from [Intro to C](../../Intro_Programming/Intro_C.ipynb) §5, except the database *refuses* to let it dangle.

Our scenario: a lab logs signal recordings from multiple sensors. Two entities → two tables:

- `sensors` — one row per physical device.
- `recordings` — one row per capture, pointing at its sensor.

In [1]:
import sqlite3

db = sqlite3.connect(":memory:")          # RAM-only; use a filename to persist
db.execute("PRAGMA foreign_keys = ON")    # SQLite needs this opt-in!

db.executescript("""
CREATE TABLE sensors (
    sensor_id   INTEGER PRIMARY KEY,
    name        TEXT NOT NULL UNIQUE,
    kind        TEXT NOT NULL CHECK (kind IN ('accelerometer', 'microphone', 'ecg')),
    fs_hz       REAL NOT NULL CHECK (fs_hz > 0)
);
CREATE TABLE recordings (
    rec_id      INTEGER PRIMARY KEY,
    sensor_id   INTEGER NOT NULL REFERENCES sensors(sensor_id),
    started_at  TEXT NOT NULL,            -- ISO 8601 timestamp
    n_samples   INTEGER NOT NULL,
    rms         REAL                      -- a computed feature, maybe NULL until processed
);
""")
print("schema created")

schema created


In [2]:
db.executemany("INSERT INTO sensors (name, kind, fs_hz) VALUES (?, ?, ?)", [
    ("acc-01", "accelerometer", 1000.0),
    ("mic-01", "microphone",    48000.0),
    ("ecg-01", "ecg",           360.0),
])

db.executemany(
    "INSERT INTO recordings (sensor_id, started_at, n_samples, rms) VALUES (?, ?, ?, ?)", [
    (1, "2026-07-20T10:00:00", 60_000,  0.42),
    (1, "2026-07-20T11:00:00", 60_000,  0.51),
    (2, "2026-07-20T10:30:00", 480_000, 0.12),
    (2, "2026-07-21T09:00:00", 960_000, 0.19),
    (3, "2026-07-21T09:30:00", 21_600,  None),   # not yet processed
])
db.commit()
print("rows:", db.execute("SELECT COUNT(*) FROM recordings").fetchone()[0])

rows: 5


**What just happened.** Three sensor rows and five recording rows went in, and `COUNT(*)` confirms 5 landed in `recordings`. Notice the last insert deliberately passes `None` for `rms` — SQL's NULL, meaning "no value yet," not zero or an empty string. That's a real state (an unprocessed recording), not missing data to paper over, and Session 2's aggregation queries will show exactly how SQL treats it differently from a numeric zero.

### 2.1. Constraints Earn Their Keep

Bad data is *rejected at the door* — a guarantee no CSV can make.

In [3]:
for bad in [
    ("INSERT INTO sensors (name, kind, fs_hz) VALUES ('x-01', 'thermometer', 10)", "bad kind"),
    ("INSERT INTO sensors (name, kind, fs_hz) VALUES ('acc-01', 'ecg', 360)",      "duplicate name"),
    ("INSERT INTO recordings (sensor_id, started_at, n_samples) VALUES (99, 't', 5)", "dangling foreign key"),
]:
    try:
        db.execute(bad[0])
    except sqlite3.IntegrityError as e:
        print(f"REJECTED ({bad[1]}): {e}")

REJECTED (bad kind): CHECK constraint failed: kind IN ('accelerometer', 'microphone', 'ecg')
REJECTED (duplicate name): UNIQUE constraint failed: sensors.name
REJECTED (dangling foreign key): FOREIGN KEY constraint failed


### 2.2. SELECT: Asking Questions

In [4]:
print("-- long recordings, newest first --")
for row in db.execute("""
    SELECT rec_id, sensor_id, started_at, n_samples
    FROM recordings
    WHERE n_samples >= 60000
    ORDER BY started_at DESC
"""):
    print(row)

print("-- unprocessed (rms IS NULL) --")
print(db.execute("SELECT rec_id FROM recordings WHERE rms IS NULL").fetchall())

-- long recordings, newest first --
(4, 2, '2026-07-21T09:00:00', 960000)
(2, 1, '2026-07-20T11:00:00', 60000)
(3, 2, '2026-07-20T10:30:00', 480000)
(1, 1, '2026-07-20T10:00:00', 60000)
-- unprocessed (rms IS NULL) --
[(5,)]


**What just happened.** `ORDER BY started_at DESC` sorted the ≥60,000-sample recordings newest-first — rec 4 (2026-07-21) before rec 2 and rec 3 (both 2026-07-20), with rec 1 last. The second query, `WHERE rms IS NULL`, is the reason SQL has a dedicated `IS NULL`/`IS NOT NULL` operator instead of letting you write `rms = NULL`: NULL means "unknown," and by SQL's three-valued logic, `NULL = NULL` itself evaluates to unknown (not true), so an ordinary `=` comparison would silently match nothing. Only rec 5 — the not-yet-processed ECG capture from the insert cell — comes back.

---
### 🕐 Session 2 of 3 — *Joins, Aggregation & Transactions* (~35 min)
**Goal:** combine tables with JOIN; summarize with GROUP BY; make multi-step changes atomic.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (databases from Python, files & analytics).

---

<details><summary>🎓 <b>Teacher notes — Session 2: Joins, Aggregation & Transactions</b></summary>

**JOIN is the payoff for having split the data into two tables in Session 1 — say this explicitly.** Splitting `sensors` and `recordings` avoided repeating each sensor's name/kind/sample-rate on every recording row (normalization), but that means answering "what's the sample rate for this recording" now requires stitching the tables back together — that's what `JOIN ... ON s.sensor_id = r.sensor_id` does. Point out the computed column `r.n_samples / s.fs_hz AS seconds` in the same query: a JOIN isn't just concatenation, you can compute with columns from both tables in one pass.

**GROUP BY is the SQL-native `reduce`, and the NumPy analogy in the intuition cell is worth leaning on hard** for this audience: rows sharing a `sensor_id` fold into one summary row, exactly like `np.sum(x, axis=0)` collapses one axis. `HAVING COUNT(*) >= 1` is deliberately a no-op filter here (every sensor has ≥1 recording) — it's included to show *where* a post-aggregation filter goes syntactically (`HAVING` filters groups, `WHERE` filters rows before grouping), which is a common point of confusion worth addressing even via a trivial example.

**The NULL-skipping in `AVG(rms)` is a small trap worth calling out loudly** (the markdown cell right after already flags it, but it's easy to skim past): aggregate functions silently ignore NULLs rather than treating them as zero or erroring. A student computing "average accuracy across runs" where some runs crashed and logged NULL will get the average of the *completed* runs, not a penalized average — sometimes what you want, sometimes a silent bug in a paper's results table.

**Transactions: frame `with db:` as "no partial states visible to the outside world."** The demo forces a crash mid-transaction and shows the update never took effect — `rec_id=1` is still on its original sensor. The teaching point isn't the syntax, it's *why this matters*: without the transaction wrapper, a crash between two dependent updates (e.g., moving a recording between sensors, which the demo's docstring alludes to) could leave the database in a state that never should have existed — recording pointing at one sensor's data but attributed to another. `ROLLBACK` (automatic here on exception) is the undo button that makes "all or nothing" enforceable rather than aspirational.
</details>

## 3. Joins & Aggregation

💡 **Intuition.** A **JOIN** re-follows the foreign-key pointers to reassemble the split entities: "for each recording, look up its sensor's row and staple them together." **GROUP BY** then folds rows sharing a key into one summary row — the SQL analogue of a NumPy `reduce` along an axis.

In [5]:
print("-- recordings with their sensor names (JOIN) --")
for row in db.execute("""
    SELECT r.rec_id, s.name, s.kind, r.n_samples / s.fs_hz AS seconds
    FROM recordings AS r
    JOIN sensors AS s ON s.sensor_id = r.sensor_id
"""):
    print(row)

-- recordings with their sensor names (JOIN) --
(1, 'acc-01', 'accelerometer', 60.0)
(2, 'acc-01', 'accelerometer', 60.0)
(3, 'mic-01', 'microphone', 10.0)
(4, 'mic-01', 'microphone', 20.0)
(5, 'ecg-01', 'ecg', 60.0)


**What just happened.** Every one of the 5 `recordings` rows came back paired with its sensor's `name` and `kind` — the JOIN re-attached the metadata that normalization had split off, using only `sensor_id` as the glue. The computed `seconds` column (`n_samples / fs_hz`) shows the duration falls out for free once the join brings the right sample rate alongside each row: the accelerometer recordings are 60 s each (60,000 samples at 1000 Hz), the microphone recordings are 10 s and 20 s (480,000 and 960,000 samples at 48,000 Hz), and the ECG recording is 60 s (21,600 samples at 360 Hz) — three different sensors, three different sample rates, one query.

In [6]:
print("-- per-sensor summary (GROUP BY) --")
for row in db.execute("""
    SELECT s.name,
           COUNT(*)            AS n_recs,
           SUM(r.n_samples)    AS total_samples,
           ROUND(AVG(r.rms), 3) AS mean_rms
    FROM recordings r
    JOIN sensors s USING (sensor_id)
    GROUP BY s.sensor_id
    HAVING COUNT(*) >= 1
    ORDER BY total_samples DESC
"""):
    print(row)

-- per-sensor summary (GROUP BY) --
('mic-01', 2, 1440000, 0.155)
('acc-01', 2, 120000, 0.465)
('ecg-01', 1, 21600, None)


Note `AVG(rms)` quietly *skipped* the NULL — aggregate functions ignore NULLs. Learn this before it surprises you in a paper's results table.

## 4. Transactions

💡 **Intuition.** A transaction makes several statements **all-or-nothing**. Classic example: moving a recording between sensors touches two rows; a crash between the two updates would corrupt the story. `BEGIN … COMMIT` means the database never shows the world a half-done state — and `ROLLBACK` is your undo.

In [7]:
try:
    with db:                                   # the connection as context manager = one transaction
        db.execute("UPDATE recordings SET sensor_id = 2 WHERE rec_id = 1")
        raise RuntimeError("power failure!")   # simulate a crash mid-transaction
except RuntimeError as e:
    print("crashed:", e)

# The update was rolled back automatically:
print("rec 1 still on sensor:", db.execute(
    "SELECT sensor_id FROM recordings WHERE rec_id = 1").fetchone()[0])

crashed: power failure!
rec 1 still on sensor: 1


**What just happened.** The `UPDATE` inside the `with db:` block ran, then the simulated crash (`raise RuntimeError`) fired before the block could complete — and the follow-up query shows `rec_id=1` is *still on sensor 1*, as if the UPDATE never happened. That's the transaction's whole job: `with db:` opened an implicit transaction, and an exception inside it triggers an automatic rollback instead of a partial commit. Nothing about the underlying UPDATE statement was wrong; what changed is that the database refused to let the outside world see a half-finished change. This is the concrete mechanism behind "the database never shows a half-done state" from the intuition cell above — not a promise, a tested behavior.

---
### 🕐 Session 3 of 3 — *Databases from Python, Safely & at Scale* (~40 min)
**Goal:** parameterized queries, storing experiment results, and when to reach beyond SQLite.
**Builds on:** Session 2.

---

<details><summary>🎓 <b>Teacher notes — Session 3: Databases from Python, Safely & at Scale</b></summary>

**Parameterized queries are the single most important security habit in this workshop — do not present this as optional style.** The demo's hostile string `"acc-01'; DROP TABLE recordings; --"` is a real SQL-injection payload: if it were pasted directly into an f-string query, the trailing `--` would comment out the rest of the original statement and the `DROP TABLE` would execute. Because `?` binds the value as *data*, SQLite never parses it as SQL syntax at all — the query just looks for a sensor literally named `acc-01'; DROP TABLE recordings; --`, finds nothing, and `recordings` survives untouched. Make the rule absolute: "never use an f-string/`.format()`/`%` to build a query with any value that isn't a hardcoded literal you wrote yourself" — there is no safe amount of string-building SQL.

**The results-logger pattern (§5.2) is the piece most likely to actually get reused** — this is worth treating as a template to copy, not just read. The shape (`experiments` table with a timestamp default, a `log_experiment()` wrapper that opens a transaction per call) generalizes directly to any parameter sweep: swap `model`/`lr`/`final_loss` for whatever hyperparameters and metrics matter. Contrast explicitly with the CSV-juggling status quo the intro section named — `ORDER BY final_loss LIMIT 1` replaces manually scanning spreadsheet rows for the best run.

**The "beyond SQLite" table is a map for later, not testable content** — the one thing worth students retaining is the *don't* in the third row: never store dense signal arrays one-sample-per-row in a relational table. Point them back at the Intro to OS discussion of pages/files if it helps: bulk numeric arrays belong in a format built for sequential bulk I/O (HDF5/Parquet/NumPy `.npy`), and SQL's job is to index and describe *where that array lives*, not to hold the samples themselves.
</details>

## 5. Python Patterns

### 5.1. Parameterized Queries — the Only Way

Never build SQL with f-strings: user input containing a quote breaks the query at best, *becomes* the query at worst (SQL injection). The `?` placeholder passes values out-of-band, so data can never be mistaken for code.

In [8]:
user_input = "acc-01'; DROP TABLE recordings; --"     # a hostile "sensor name"

# SAFE: value is bound as data — no row matches, nothing else happens
rows = db.execute("SELECT * FROM sensors WHERE name = ?", (user_input,)).fetchall()
print("safe query matched:", rows)
print("recordings table still exists:",
      db.execute("SELECT COUNT(*) FROM recordings").fetchone()[0], "rows")

safe query matched: []
recordings table still exists: 5 rows


**What just happened.** The hostile string went in as an ordinary bound parameter, and two things prove the defense worked: the query matched *zero* rows (no sensor is literally named `acc-01'; DROP TABLE recordings; --`, so nothing was found — the embedded quote and `--` comment marker were never interpreted as SQL), and `recordings` still holds all 5 rows afterward. Had this same string been spliced into the query text with an f-string instead of bound with `?`, the quote would have closed the string literal early and the `DROP TABLE recordings` clause would have executed as a second statement — this cell is a working demonstration of the exploit being neutralized, not just an assertion that parameterization is safe.

### 5.2. A Reusable Results Logger

The pattern worth stealing: every training run / parameter sweep in your research logs into a table, and analysis is a query away — compare with juggling 40 `results_final_v2 (copy).csv` files.

In [9]:
db.executescript("""
CREATE TABLE IF NOT EXISTS experiments (
    exp_id     INTEGER PRIMARY KEY,
    ran_at     TEXT DEFAULT (datetime('now')),
    model      TEXT NOT NULL,
    lr         REAL NOT NULL,
    final_loss REAL NOT NULL
);
""")

def log_experiment(model, lr, final_loss):
    with db:
        db.execute("INSERT INTO experiments (model, lr, final_loss) VALUES (?, ?, ?)",
                   (model, lr, final_loss))

# pretend sweep — in real life these come from your training loop
for lr, loss in [(0.1, 0.083), (0.01, 0.041), (0.001, 0.067)]:
    log_experiment("mlp-32", lr, loss)

print("best run:", db.execute(
    "SELECT model, lr, final_loss FROM experiments ORDER BY final_loss LIMIT 1").fetchone())

best run: ('mlp-32', 0.01, 0.041)


**What just happened.** Three pretend training runs at learning rates 0.1, 0.01, and 0.001 logged their final losses (0.083, 0.041, 0.067) into the `experiments` table via `log_experiment()`, and a single `ORDER BY final_loss LIMIT 1` query picked out the winner: `lr=0.01` at loss 0.041. Notice the U-shape in the three losses — lr=0.1 too aggressive (0.083), lr=0.01 close to optimal (0.041), lr=0.001 too slow to converge within a fixed budget (0.067) — the classic learning-rate sweet spot, except here it's *queryable* rather than something you'd have to remember from scrolling through three separate log files.

### 5.3. Beyond SQLite

| Need | Reach for |
|---|---|
| Many concurrent writers, network access, roles | **PostgreSQL** — same SQL, industrial engine |
| Analytics over millions of rows / Parquet files | **DuckDB** — SQLite's analytical twin |
| Long dense signal data | Store *arrays* in files (HDF5/Parquet), *metadata + paths* in SQL — don't put 48 kHz samples one-per-row |

The SQL you wrote today transfers to all of them nearly verbatim.

## 6. Conclusion

You modeled entities as tables, protected them with constraints, asked real questions with joins and aggregation, made changes atomic, and built the experiment-logging habit. That's 90% of the database craft a signals/ML researcher needs.

---
## Where next

- [Intro to Operating Systems](../README.md#workshop-1--introduction-to-operating-systems-available) — the files, processes, and locks a database is built from.
- [Intro to Python](../../Intro_Programming/Intro_Python/Intro_Python.ipynb) — the NumPy side of the "arrays in files, metadata in SQL" pattern.